## Data Download and split

Goal: Get the raw data onto disk and split into train/eval and holdout(test)

In [5]:
import pandas as pd 
from sklearn.model_selection import train_test_split 
from src.feature_pipeline.load import download_raw_data
from src.utils.paths import RAW_DIR

In [6]:
raw_path = download_raw_data()
raw_path

100%|██████████| 172k/172k [00:00<00:00, 545kB/s]

Extracting files...
✅ Downloaded WA_Fn-UseC_-Telco-Customer-Churn.csv -> /home/teehy/ml_class/churn-prediction-mle/data/raw/telco_churn.csv


PosixPath('/home/teehy/ml_class/churn-prediction-mle/data/raw/telco_churn.csv')

In [7]:
df = pd.read_csv(raw_path)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [8]:
df.shape

(7043, 21)

In [9]:
df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


## Split method: Stratified random split

We use a **stratified random split** instead. Only about a quarter of customers churn, so stratifying on `Churn`
keeps the same churn rate in every set. Without it, one set could end up with noticeably fewer churners than another.

| Set | Share | Used for |
|---|---|---|
| train | 70% | learning, EDA, feature fitting |
| eval | 15% | model selection and hyperparameter tuning |
| holdout | 15% | touched **once**, at the very end |

*Real life note:* with timestamps (for example monthly snapshots), we would split by time so the model is always tested on the future.

In [11]:
df['Churn'].value_counts(normalize=True)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

In [13]:
SEED = 42

train_df, temp_df = train_test_split(
    df, train_size = 0.70, stratify=df['Churn'], random_state=SEED
)

eval_df, test_df = train_test_split(
    temp_df, test_size = 0.50, stratify=temp_df['Churn'], random_state=SEED
)


In [14]:
holdout_df = test_df

In [15]:
for name, part in [("train", train_df), ("eval", eval_df), ("holdout", holdout_df)]:
    churn_rate = (part['Churn']=="Yes").mean()
    print(f"{name:8s} rows={len(part):5d} churn rate ={churn_rate:.1%}")

train    rows= 4930 churn rate =26.5%
eval     rows= 1056 churn rate =26.5%
holdout  rows= 1057 churn rate =26.6%


In [16]:
# No customer in more than one set, and nothing lost
assert len(train_df) + len(eval_df) + len(holdout_df) == len(df)
assert set(train_df["customerID"]).isdisjoint(eval_df["customerID"])
assert set(train_df["customerID"]).isdisjoint(holdout_df["customerID"])
assert set(eval_df["customerID"]).isdisjoint(holdout_df["customerID"])
print("✅ splits are disjoint and complete")

✅ splits are disjoint and complete


In [17]:
expected = len(train_df) + len(eval_df) + len(holdout_df)

In [18]:
expected

7043

In [19]:
len(df)

7043

In [21]:
train_df.to_csv(RAW_DIR / "train.csv", index =False)
eval_df.to_csv(RAW_DIR / "eval.csv", index=False)
holdout_df.to_csv(RAW_DIR / "holdout_df", index=False)